In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib
import torch
import torch.nn as nn

ROOT = Path().resolve().parent.parent
DATA_PROCESSED = ROOT / 'data' / 'processed'

X_train_sc = np.load(DATA_PROCESSED / "X_train.npy")
X_val_sc   = np.load(DATA_PROCESSED / "X_val.npy")
X_test_sc  = np.load(DATA_PROCESSED / "X_test.npy")

Y_train = pd.read_parquet(DATA_PROCESSED / "Y_train.parquet")
Y_val   = pd.read_parquet(DATA_PROCESSED / "Y_val.parquet")
Y_test  = pd.read_parquet(DATA_PROCESSED / "Y_test.parquet")

X = pd.read_parquet(DATA_PROCESSED / 'X.parquet')
Y = pd.read_parquet(DATA_PROCESSED / 'Y.parquet')

## Definition du modèle MLP

In [ ]:


class MLP(nn.Module):
    """
    MLP multi-sorties (une sortie par cible Y).
    
    Args:
        input_dim  : nb de features (X_train.shape[1]) 110 attributs
        output_dim : nb de cibles   (len(Y.columns)) 5 cibles
        hidden_dims: tuple de la taille de chaque couche cachée
        dropout    : taux de dropout (régularisation)
    """
    def __init__(self, input_dim, output_dim,
                 hidden_dims=(256, 128, 64),
                 dropout=0.2):
        super().__init__()

        layers = []
        in_dim = input_dim

        for h in hidden_dims:
            layers += [
                nn.Linear(in_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            in_dim = h

        layers.append(nn.Linear(in_dim, output_dim))  # pas d'activation en sortie

        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

# Instanciation
input_dim  = X_train_sc.shape[1]   
output_dim = len(Y.columns)      

model = MLP(
    input_dim   = input_dim,
    output_dim  = output_dim,
    hidden_dims = (256, 128, 64),
    dropout     = 0.2,
)

print(model)
print(f"\nParamètres entraînables : {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


## Loss et Optimizer

In [ ]:
# Loss :mesurer l'écart entre les valeurs réelles et les prédictions d'un modèle de régression.
# on utilise HuberLoss : combinaison de MSE et MAE  

criterion = nn.HuberLoss(delta=1.0)   # delta à ajuster selon l'échelle de Y

# Optimizer ─
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr           = 1e-3,
    weight_decay = 1e-4,   # L2 régularisation intégrée
)

# réduit le lr si la val_loss est stable
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=10,
)
# Vérification
print(f"Optimizer : {optimizer.__class__.__name__}")
print(f"LR initial : {optimizer.param_groups[0]['lr']}")



## DataLoader

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Conversion numpy à tenseur et création de DataLoaders
def make_loader(X_np, Y_df, batch_size, shuffle=False):
    X_t = torch.tensor(X_np, dtype=torch.float32)
    Y_t = torch.tensor(Y_df.values, dtype=torch.float32)
    ds  = TensorDataset(X_t, Y_t)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                       num_workers=0, pin_memory=False)

# Loaders 
BATCH_SIZE = 512

train_loader = make_loader(X_train_sc, Y_train, BATCH_SIZE, shuffle=True) #batch
val_loader   = make_loader(X_val_sc,   Y_val,   BATCH_SIZE, shuffle=False)
test_loader  = make_loader(X_test_sc,  Y_test,  BATCH_SIZE, shuffle=False)

print(f"Train : {len(train_loader.dataset):,} samples -> {len(train_loader)} batches") #384 979 / 512 = 752 batches
print(f"Val   : {len(val_loader.dataset):,} samples")
print(f"Test  : {len(test_loader.dataset):,} samples")


In [ ]:
import copy, time

# Early Stopping : arrêter l'entraînement si la val_loss ne s'améliore pas pendant un certain nombre d'epochs
class EarlyStopping:
    def __init__(self, patience=20, delta=1e-4):
        self.patience  = patience
        self.delta     = delta
        self.counter   = 0
        self.best_loss = float('inf')
        self.best_weights = None

    def step(self, val_loss, model):
        if val_loss < self.best_loss - self.delta:
            self.best_loss    = val_loss
            self.best_weights = copy.deepcopy(model.state_dict())
            self.counter      = 0
            return False   
        self.counter += 1
        return self.counter >= self.patience 

    def restore(self, model):
        model.load_state_dict(self.best_weights)


# Boucle d'entraînement 
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for X_b, Y_b in loader:
        X_b, Y_b = X_b.to(device), Y_b.to(device)
        optimizer.zero_grad()
        pred = model(X_b)
        loss = criterion(pred, Y_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(X_b)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for X_b, Y_b in loader:
            X_b, Y_b = X_b.to(device), Y_b.to(device)
            pred       = model(X_b)
            total_loss += criterion(pred, Y_b).item() * len(X_b)
    return total_loss / len(loader.dataset)


# Entraînement principal
N_EPOCHS   = 200
device     = "cuda" if torch.cuda.is_available() else "cpu"
model      = model.to(device)
early_stop = EarlyStopping(patience=20, delta=1e-4)
history    = {'train': [], 'val': []}

print(f"Entraînement sur : {device}\n")
t0 = time.time()

for epoch in range(1, N_EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss   = evaluate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    history['train'].append(train_loss)
    history['val'].append(val_loss)

    if epoch % 10 == 0:
        elapsed = time.time() - t0
        print(f"Epoch {epoch:>4}/{N_EPOCHS}  "
              f"train={train_loss:.4f}  val={val_loss:.4f}  "
              f"({elapsed:.0f}s)")

    if early_stop.step(val_loss, model):
        print(f"\n⏹ Early stop à l'epoch {epoch}")
        break

# Restaure les meilleurs poids
early_stop.restore(model)
print(f"\n✓ Meilleure val_loss : {early_stop.best_loss:.4f}")

cfg = {
    "batch_size": 512,
    "lr": 1e-3,
    "hidden_dims": (256, 128, 64),
    "dropout": 0.2,
    "loss": "HuberLoss",
}

# Sauvegarde
torch.save({
    'model_state': model.state_dict(),
    'history': history,
    'config': cfg,
}, 'best_model.pt')

In [ ]:

checkpoint = torch.load("best_model.pt", map_location="cpu")

print(type(checkpoint))
print(checkpoint.keys())